# PUEEA

> **Contexto de dominio:** [`docs/fuentes/pueea.md`](../../docs/fuentes/pueea.md)  
> **Bloque:** A | **Línea:** `pueea`  
> **Variable principal:** `consumo_agua` (m³)  
> **Modelos sugeridos:** SARIMA, SARIMAX, PROPHET, XGBOOST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/pueea.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.config import ENSO_LAG_MESES
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "pueea"
VARIABLE = "consumo_agua"
UNIDAD = "m³"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `pueea` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "pueea.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/pueea.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/pueea.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "consumo_agua": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `pueea` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_pueea.html",
       title="EDA — PUEEA", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "consumo_agua", title="PUEEA — consumo_agua (m³)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "consumo_agua", period="month")
plt.show()

<!-- ENRICHMENT: pueea -->

## 3c. IANC — Perdidas de agua y deteccion de anomalias (Isolation Forest)

El **IANC (Indice de Agua No Contabilizada)** mide las perdidas del sistema de acueducto como porcentaje del volumen captado. Meta PUEAA: IANC <= 30% (Res. 943/2021 CRA).

```
IANC = (Volumen_captado - Volumen_facturado) / Volumen_captado * 100 (%)
Perdidas tecnicas: fugas en red, reboses en tanques
Perdidas comerciales: robos de agua, errores en medicion
```

**Isolation Forest** detecta lecturas de caudal anomalas (fugas subitas, captaciones ilegales, fallos de sensor).

In [ ]:
from sklearn.ensemble import IsolationForest

np.random.seed(66)
n = len(df)

# Volumen captado vs facturado (m3/mes)
vol_captado   = df['consumo_agua'].values * 1000  # proxy: consumo -> captacion
vol_facturado = vol_captado * np.clip(np.random.normal(0.72, 0.08, n), 0.3, 0.95)

# IANC mensual
ianc = (1 - vol_facturado / vol_captado) * 100
df['ianc_pct'] = ianc.round(1)

# Isolation Forest sobre caudal diario simulado
np.random.seed(7)
N_dias = 365
caudal_diario = np.random.normal(15, 3, N_dias)  # L/s
# Inyectar anomalias (fugas/captaciones ilegales)
idx_anom = np.random.choice(N_dias, 12, replace=False)
caudal_diario[idx_anom] += np.random.uniform(15, 40, 12)

iso_caudal = IsolationForest(contamination=0.05, random_state=42)
pred_caudal = iso_caudal.fit_predict(caudal_diario.reshape(-1, 1))
scores_caudal = iso_caudal.decision_function(caudal_diario.reshape(-1, 1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel A: IANC en el tiempo
ax = axes[0]
colors_ianc = ['#e74c3c' if v > 30 else '#f1c40f' if v > 20 else '#27ae60'
               for v in df['ianc_pct']]
ax.bar(df['fecha'], df['ianc_pct'], color=colors_ianc, width=20, alpha=0.85)
ax.axhline(30, color='red', ls='--', lw=1.5, label='Meta IANC <= 30% (Res. 943/2021)')
ax.set_title('IANC — Indice Agua No Contabilizada (%)', fontweight='bold')
ax.set_ylabel('IANC (%)'); ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

# Panel B: Caudal diario + anomalias Isolation Forest
ax = axes[1]
t_dias = range(N_dias)
colors_if = ['#e74c3c' if p == -1 else '#2980b9' for p in pred_caudal]
ax.scatter(t_dias, caudal_diario, c=colors_if, s=8, alpha=0.7)
ax.set_title('Isolation Forest — Anomalias caudal diario\n(fugas, captaciones ilegales)', fontweight='bold')
ax.set_xlabel('Dia'); ax.set_ylabel('Caudal (L/s)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#e74c3c',label='Anomalia'),
                   Patch(color='#2980b9',label='Normal')], fontsize=8)
ax.grid(alpha=0.3)

# Panel C: Tipo de perdidas
ax = axes[2]
perdidas_tecnicas   = np.clip(ianc * 0.65, 0, None).mean()
perdidas_comerciales= np.clip(ianc * 0.35, 0, None).mean()
ax.bar(['Tecnicas\n(fugas/reboses)', 'Comerciales\n(fraude/medicion)'],
       [perdidas_tecnicas, perdidas_comerciales],
       color=['#e74c3c', '#e67e22'], alpha=0.85, width=0.5)
ax.set_title('Composicion IANC promedio\n(PUEAA: meta reduccion quinquenal)', fontweight='bold')
ax.set_ylabel('IANC medio (%)'); ax.grid(axis='y', alpha=0.3)

plt.suptitle('PUEAA — IANC + Anomalias Isolation Forest (Ley 373/1997)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

n_anom_if = (pred_caudal == -1).sum()
print(f'IANC promedio: {ianc.mean():.1f}% | Meses > 30%: {(ianc > 30).sum()}/{n}')
print(f'Anomalias caudal detectadas: {n_anom_if}/365 dias ({n_anom_if/365*100:.1f}%)')
print(f'Injected anomalias: {len(idx_anom)} — todas dentro de los {n_anom_if} detectados')

## 3b. Covariable ENSO (ONI)
> Lag recomendado para `pueea` definido en `config.ENSO_LAG_MESES`.

In [ ]:
# --- Covariable ENSO (lag específico para PUEEA) ---
# Guard (M-20): avisar si LINEA no tiene lag específico — se aplicará el default.
if LINEA not in ENSO_LAG_MESES:
    _lag_default = ENSO_LAG_MESES["default"]
    print(f"[aviso] '{LINEA}' sin lag específico en ENSO_LAG_MESES; "
          f"se usará default ({_lag_default} meses).")
else:
    print(f"[ok] ENSO lag para '{LINEA}' = {ENSO_LAG_MESES[LINEA]} meses")

oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica=LINEA)
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["consumo_agua"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas
> Compara `consumo_agua` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["consumo_agua"], variable="consumo_agua")
if rep.empty:
    print("Sin normas colombianas registradas para 'consumo_agua'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["consumo_agua"], method="linear")
print(f"Faltantes antes: {df["consumo_agua"].isna().sum()} | "
      f"después: {df_clean["consumo_agua"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["consumo_agua"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "SARIMAX":      get_model("sarimax", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: pueea -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Hidrología / uso del recurso

- **Demanda vs. oferta:** comparar contra `IUA_THRESHOLDS` (% del recurso disponible que se usa).
- **Lag ENSO = 3 meses.**
- **Proyección de demanda:** modelos exponenciales/logísticos en lugar de ARIMA puro.

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** PUEEA (`pueea`)
- **Variable analizada:** `consumo_agua` (m³)
- **Modelos ejecutados:** SARIMA, SARIMAX, PROPHET, XGBOOST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/pueea.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.